# Open web data for everyone: The Common Crawl corpora

Welcome! This notebook is an introduction to Common Crawl's corpora and how you can access and use them. In this session, we will cover:

* What is the Common Crawl Foundation?
* The corpora
* Different ways to access 
* Useful Python tools for interacting with the data
* Example uses

This notebook contains both links to web pages (often Common Crawl's website) and Python example code you can run. We have tried to keep the size of data downloaded in this notebook's examples reasonably small. If you want to access larger amounts of Common Crawl data, we provide links and signposts to where you can find out more 🚛

## What is the Common Crawl Foundation?

[<img src="Common_Crawl_Logo_White.svg" width="400"/>](https://commoncrawl.org/)

The Common Crawl Foundation is a US 501(c)(3) non-profit, founded in 2007 by Gil Elbaz. Its mission is to make high-quality web crawl data available to anyone who wants it, not just big companies. It's run by a small (but growing) team of staff and volunteers, with most of the engineers (including [me!](https://commoncrawl.org/team/laurie-burchell)) based in Europe.

The core of Common Crawl's offering is a **free, open repository of web crawl data.** The dataset contains more than 300 billion pages dating back to 2007, with roughly 3-5 billion pages added each month. All of our datasets are free for anyone to access, with the cost of storage covered by the [AWS Open Datasets Program](https://aws.amazon.com/opendata/open-data-sponsorship-program/).

## What's in each crawl?

Each crawl archive is made up of files which contain a number of different types of crawl data. Because it is so large, the crawl data is partitioned into multiple smaller files. To assist with exploring and using the dataset, Common Crawl provides gzipped files which list all segments, WARC, WAT and WET files. You can find these through Common Crawl's website.

Navigate to <https://commoncrawl.org/overview> and click on the dropdown to see the crawl archives:

[<img src="crawl_dropdown.png" width="600"/>](https://commoncrawl.org/overview)

The naming scheme is "CC-MAIN-YYYY-WW" (where WW is approximately the week the crawl started fetching). 

Click on one of the newer crawls near the top: in this notebook, we chose "CC-MAIN-2025-33". You should get a summary table, listing the available data types for the crawl archive you selected:

[<img src="august_2025_summary.png" width="600"/>](https://data.commoncrawl.org/crawl-data/CC-MAIN-2025-33/index.html)

As you can see from the table, each crawl has multiple data types available. Here is a brief description of those available in recent crawls, with a ⭐️ next to those this notebook focuses on. Note that files available from the hyperlinks in this table contain paths to the data, not the data itself! You will need to download the data using S3 or via HTTP from the paths given.

* **Segments**: a list of the file paths of all the segments in the crawl. 
* ⭐️ **WARC:** the raw crawl data stored in the [WARC format](https://iipc.github.io/warc-specifications/specifications/warc-format/warc-1.0/).
* **WAT:** metadata associated with the crawled web pages.
* **WET:** the body text of web pages extracted from the HTML, excluding any HTML code, images, or other media. Handy for text-based tasks.
* **Robots.txt files:** the [robots.txt](https://en.wikipedia.org/wiki/Robots.txt) pages from the sites crawled. [Common Crawl's crawler](https://commoncrawl.org/ccbot) is very polite and always respects robots.txt!
* **Non-200 responses:** A record of responses from crawled sites with [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes) other than 200 OK. 
* ⭐️ **URL index files:** A list of the file paths of the CDX index of the crawl. 
* ⭐️ **Columnar URL index files:** A list of the file paths of the columnar index of the crawl.

*N.B.: older crawls may not have all of these data types.*

## Accessing the data

There are two main ways to access the data:

1) via HTTP(S)
2) via the AWS cloud

For the sake of avoiding AWS account issues, this notebook downloads the data via HTTP(S). That said, using AWS can be pretty convenient and allows easy access to compute. See our [Get Started](https://commoncrawl.org/get-started) guide for more information on how to use the AWS platform to interact with our data.

You may have noticed in the data listing above that there are two types of index file. This is to allow different kinds of access. Let's look at both of them and how you can interact with them programatically.

### [Fetching](https://www.youtube.com/watch?v=Pubd-spHN-0) a page with the CDX(J) index

The [CDX(J) index](https://specs.webrecorder.net/cdxj/0.1.0/) is one of the two indexes of Common Crawl's data. It's useful for querying individual pages. You can find the API at <https://index.commoncrawl.org/>: try running some queries through the web interface!

Below, you'll find an example querying the CDX(J) index for a particular crawl for a single web page using Python. Try changing `index_name` and `target_url`. 

In [4]:
# imports and variables
import requests
import json
from urllib.parse import quote_plus
from warcio.archiveiterator import ArchiveIterator
from bs4 import BeautifulSoup

# you can change the following variables!
index_name = 'CC-MAIN-2025-33'  # The crawl you want to query
target_url = 'en.wikipedia.org/wiki/Byte'  # the URL you want to look up

# It’s advisable to use a descriptive User-Agent string when developing your own applications.
# This practice aligns with the conventions outlined in RFC 7231. Let's use this simple one:
myagent = 'cc-get-started/1.0 (Example data retrieval script; yourname@example.com)'

We use two functions to fetch a page using the index: one to query the index and return the location of the files (`search_cc_index`) and the other to fetch and process the page we want (`fetch_page_from_cc`).

We're using the [warcio](https://github.com/webrecorder/warcio) library to read the WARC records.

In [5]:
def search_cc_index(url, index_name, server='http://index.commoncrawl.org/'):
    """queries the Common Crawl index for a given URL and returns the records"""
    encoded_url = quote_plus(url)
    index_url = f'{server}{index_name}-index?url={encoded_url}&output=json'
    response = requests.get(index_url, headers={'user-agent': myagent})
    print("Response from server:\r\n", response.text)
    if response.status_code == 200:
        records = response.text.strip().split('\n')
        return [json.loads(record) for record in records]
    else:
        return None


def fetch_page_from_cc(records):
    """fetches the page content from Common Crawl given the index records"""
    for record in records:
        s3_url = f'https://data.commoncrawl.org/{record["filename"]}'  
        # Define the byte range for the request
        offset, length = int(record['offset']), int(record['length'])
        byte_range = f'bytes={offset}-{offset+length-1}'

        # Send the HTTP GET request to the S3 URL with the specified byte range
        response = requests.get(
            s3_url,
            headers={'user-agent': myagent, 'Range': byte_range},
            stream=True  # gzipped compressed data needs a raw byte stream
        )

        if response.status_code == 206:
            stream = ArchiveIterator(response.raw)  # handles gzipped WARC content
            for warc_record in stream:
                if warc_record.rec_type == 'response':
                    return warc_record.content_stream().read()
        else:
            print(f"Failed to fetch data: {response.status_code}")
            return b''
    print("No valid WARC record found in the given records")
    return b''

In [6]:
# use the functions and search the index for the target URL
records = search_cc_index(target_url, index_name)
if records:
    print(f"Found {len(records)} records for {target_url}")

    # Fetch the page content from the first record
    content = fetch_page_from_cc(records)
    if content:
        print(f"Successfully fetched content for {target_url}")
        # You can now process the 'content' variable using something like Beautiful Soup, etc
else:
    print(f"No records found for {target_url}")

Response from server:
 {"urlkey": "org,wikipedia,en)/wiki/byte", "timestamp": "20250813202703", "url": "https://en.wikipedia.org/wiki/BYTE", "mime": "text/html", "mime-detected": "text/html", "status": "200", "digest": "XFL7X34C5BMNVTA7OL7C34KWYCFAPZFD", "length": "32354", "offset": "171112387", "filename": "crawl-data/CC-MAIN-2025-33/segments/1754151572092.97/warc/CC-MAIN-20250813183110-20250813213110-00200.warc.gz", "languages": "eng", "encoding": "UTF-8"}

Found 1 records for en.wikipedia.org/wiki/Byte
Successfully fetched content for en.wikipedia.org/wiki/Byte


In [7]:
# let's see what the content looks like
soup = BeautifulSoup(content, 'html.parser')
print(soup.prettify()[2000:3000]) 

":1304238981,"wgRevisionId":1304238981,"wgArticleId":33564897,"wgIsArticle":true,"wgIsRedirect":false,"wgAction":"view","wgUserName":null,"wgUserGroups":["*"],"wgCategories":["Articles with short description","Short description is different from Wikidata","Use mdy dates from June 2020","Webarchive template wayback links","All articles with unsourced statements","Articles with unsourced statements from February 2022","CS1 maint: postscript","Commons category link is on Wikidata","Articles with Internet Archive links","Articles with French-language sources (fr)","Monthly magazines published in the United States","Defunct computer magazines published in the United States","Home computer magazines","Magazines disestablished in 1998","Magazines established in 1975","Defunct magazines published in New Hampshire","Informa brands"],"wgPageViewLanguage":"en","wgPageContentLanguage":"en","wgPageContentModel":"wikitext","wgRelevantPageName":"Byte_(magazine)","wgRelevantArticleId":33564897,"wgIsPr

### Fetching data with the columnar index

The second index provided by Common Crawl is the columnar index.  This format of the index is suited to efficient analytical and/or bulk queries of the data, saving time and computing resources. You can access it here: <https://data.commoncrawl.org/cc-index/table/cc-main/index.html>.

The Parquet files of the columnar index are compatible with many analytical libraries and toolkits, such as AWS Athena, Apache Spark, Pandas, Polars, Apache Arrow, DuckDB and more. Using them, you can run queries like the one below, which finds multilingual content in the `.va` TLD:


```sql
SELECT url_host_registered_domain AS domain,
    COUNT(DISTINCT(url_path_lang)) as n_lang,
    COUNT(*) as n_pages,
    histogram(url_path_lang) as lang_counts
FROM "ccindex"."ccindex",
UNNEST(regexp_extract_all(url_path, '(?<=/)(?:[a-z][a-z])(?=/)')) AS t (url_path_lang)
WHERE crawl = 'CC-MAIN-2025-33'
AND subset = 'warc'
AND url_host_registry_suffix = 'va'
GROUP BY url_host_registered_domain
HAVING COUNT(*) >= 100
AND COUNT(DISTINCT(url_path_lang)) >= 1
ORDER BY n_pages DESC;
```


Whilst querying the columnar index directly on S3 is free, it does require authenticated access. This means we won't be querying it in this notebook. If there's time in this session, I can show you how to run queries using AWS Athena! There is also an [introduction and tutorial](https://commoncrawl.org/blog/index-to-warc-files-and-urls-in-columnar-format) available on our website which uses Amazon Athena. We also have examples using [pyspark](https://github.com/commoncrawl/cc-pyspark) which can be run locally.

## Example uses

Common Crawl's data has been used in all kinds of projects. Following are just a few examples:

* Creating a large-scale, multilingual corpus for pre-training large language models ([FineWeb2](https://arxiv.org/pdf/2506.20920))
* Analysing how links to online content have disappeared over time ([When Online Content Disappears](https://www.pewresearch.org/wp-content/uploads/sites/20/2024/05/pl_2024.05.17_link-rot_report.pdf))
* Detecting misinformation sources amongst news domains ([Detection and Discovery of Misinformation Sources Using Attributed Webgraphs](https://ojs.aaai.org/index.php/ICWSM/article/view/31309/33469))
* Studying censorship of Amazon products ([Banned Books](https://citizenlab.ca/2024/11/analysis-of-censorship-on-amazon-com/))
* Building a dashboard showing news sentiment about the COVID-19 pandemic ([A COVID-19 news coverage mood map of Europe](https://aclanthology.org/2021.hackashop-1.15.pdf))

## But wait, there's more!

This is the end of the notebook! There's more that Common Crawl offers that I wasn't able to cover today: here are some noteworthy mentions:

- We have a [Whirlwind Tour](https://github.com/commoncrawl/whirlwind-python) which takes you through what we covered in this notebook in more depth.
- Our github repository has many projects and libaries using our data: <https://github.com/commoncrawl/>
- We build web graphs from our crawl data, showing the structure of the web: <https://commoncrawl.org/web-graphs>
- We have a [Discord server](https://discord.com/invite/njaVFh7avF): join to stay in touch!

Thank you!
